<a href="https://colab.research.google.com/github/gavin-hecke/Gavin_INFO5731_Spring2026/blob/main/Assignment_2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. Data Quality Check

1.1 Using Python (pandas, matplotlib, or seaborn), load and inspect the Assignment 2 dataset.

In [1]:
# Write your code here

from google.colab import files
uploaded = files.upload()


Saving Assignment 2 dataset.csv to Assignment 2 dataset.csv


Write code to explore the data distribution (e.g., region, type, year) and check whether there is any bias. Provide both the code and your interpretation.

In [2]:
# Write your code here
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('Assignment 2 dataset.csv')

# Basic info
print(df.head())
print(df.info())
print(df.describe())

# Distribution checks
print(df['region'].value_counts())
print(df['type'].value_counts())
print(df['year'].value_counts())


# The dataset appears fairly balanced across regions, types, and years.
# The only werid Thing I see is in the years their is a 1904 entry
# and a limited number of 2018 entries compared to others



   Column 1        Date  AveragePrice  Total Volume     4046       4225  \
0         0  12-27-2015          1.33      64236.62  1036.74   54454.85   
1         1  12-20-2015          1.35      54876.98   674.28   44638.81   
2         2  12-13-2015          0.93     118220.22   794.70  109149.67   
3         3   12-6-2015          1.08      78992.15  1132.00   71976.41   
4         4  11-29-2015          1.28      51039.60   941.48   43838.39   

     4770  Total Bags  Small Bags  Large Bags  XLarge Bags          type  \
0   48.16     8696.87     8603.62       93.25          0.0  conventional   
1   58.33     9505.56     9408.07       97.49          0.0  conventional   
2  130.50     8145.35     8042.21      103.14          0.0  conventional   
3   72.58     5811.16     5677.40      133.76          0.0  conventional   
4   75.78     6183.95     5986.26      197.69          0.0  conventional   

   year  region  
0  2015  Albany  
1  2015  Albany  
2  2015  Albany  
3  2015  Albany  
4 

1.2 Write Python code to check for duplicate rows and missing values in the dataset. Show the number of duplicates and missing values for each column. Then, explain (in comments or markdown) how you would handle these issues (e.g., drop, impute, or replace).

In [3]:
# Write your code here
print("Duplicate rows:", df.duplicated().sum())
print("Missing values per column:\n", df.isnull().sum())

# Handling strategy: Drop duplicate rows, For missing values: impute with median per region
# If row has many missing values, drop the row

Duplicate rows: 2
Missing values per column:
 Column 1        0
Date            0
AveragePrice    0
Total Volume    1
4046            2
4225            1
4770            1
Total Bags      1
Small Bags      2
Large Bags      2
XLarge Bags     1
type            1
year            0
region          0
dtype: int64


1.3 Use Python code to print the number of rows and columns in the dataset (e.g., with df.shape). Based on the dataset size, explain (briefly) whether you think the dataset is sufficient for training a machine learning model.

In [4]:
# Write your code here
print("Shape:", df.shape)

# The dataset size is large enough for basic ML models,
# but more complex models may require more data.

Shape: (18254, 14)


#2. Data Cleaning and Preprocessing

2.1 Remove the first column or “Column 1” from the dataset. Treat the ‘year’ variable as nominal.

In [8]:
# Write your code here
df = df.drop(columns=['Column 1'], errors='ignore')
df['year'] = df['year'].astype('category')

2.2 Check for duplicate values and remove them.

In [6]:
# Write your code here
df = df.drop_duplicates()
print("Duplicates after removal:", df.duplicated().sum())

Duplicates after removal: 0


2.3 Check for missing values. If a data record (row) only has a few missing values, replace the missing values with the median of the column feature in that specific “Region” variable. If most column values in a data record are missing, remove the data record.

In [9]:
# Write your code here
df = df[df.isnull().mean(axis=1) < 0.5]

# Fill remaining missing values with median per region
for col in df.select_dtypes(include='number').columns:
    df[col] = df.groupby('region')[col].transform(
        lambda x: x.fillna(x.median())
    )

print(df.isnull().sum())

Date            0
AveragePrice    0
Total Volume    0
4046            0
4225            0
4770            0
Total Bags      0
Small Bags      0
Large Bags      0
XLarge Bags     0
type            0
year            0
region          0
dtype: int64


2.4 Find the correlation between the variables and describe how the correlated values among the variables impact the model accuracy.


In [10]:
# Write your code here
corr = df.select_dtypes(include='number').corr()
print(corr)

plt.show()

# Total Volume and bag/PLU variables (4046, 4225, 4770, Bags) are extremely highly correlated (~0.9+),
# indicating strong multicollinearity. This can hurt linear model stability and interpretability.
# AveragePrice shows weak correlation with volume features, so price prediction may require more features.


              AveragePrice  Total Volume      4046      4225      4770  \
AveragePrice      1.000000     -0.192767 -0.208325 -0.172944 -0.179458   
Total Volume     -0.192767      1.000000  0.977863  0.974181  0.872203   
4046             -0.208325      0.977863  1.000000  0.926110  0.833390   
4225             -0.172944      0.974181  0.926110  1.000000  0.887856   
4770             -0.179458      0.872203  0.833390  0.887856  1.000000   
Total Bags       -0.177103      0.963047  0.920057  0.905788  0.792315   
Small Bags       -0.174742      0.967238  0.925280  0.916032  0.802734   
Large Bags       -0.172953      0.880640  0.838646  0.810016  0.698473   
XLarge Bags      -0.117604      0.747158  0.699378  0.688810  0.679862   

              Total Bags  Small Bags  Large Bags  XLarge Bags  
AveragePrice   -0.177103   -0.174742   -0.172953    -0.117604  
Total Volume    0.963047    0.967238    0.880640     0.747158  
4046            0.920057    0.925280    0.838646     0.699378  
422

#3. Exploratory Data Analysis (EDA)


3.1 Describe the variables
- Describe all variables in the dataset.
- For continuous variables: report **range (min, max), mean, median, and distribution**.
- For categorical variables: list unique values.

In [11]:
# Write your code here
for col in df.select_dtypes(include='number').columns:
    print(f"\n{col}")
    print("Min:", df[col].min())
    print("Max:", df[col].max())
    print("Mean:", df[col].mean())
    print("Median:", df[col].median())

# Categorical variables
for col in df.select_dtypes(include=['object','category']).columns:
    print(f"\n{col} unique values:")
    print(df[col].unique())


AveragePrice
Min: 0.44
Max: 3.25
Mean: 1.406020492027834
Median: 1.37

Total Volume
Min: 84.56
Max: 62505646.52
Mean: 850552.3121034465
Median: 107354.25

4046
Min: 0.0
Max: 22743616.17
Mean: 292983.9537370555
Median: 8645.3

4225
Min: 0.0
Max: 20470572.61
Mean: 295122.54683140654
Median: 29056.73

4770
Min: 0.0
Max: 2546439.11
Mean: 22837.27302997096
Median: 184.99

Total Bags
Min: 0.0
Max: 19373134.37
Mean: 239613.96402060156
Median: 39738.53

Small Bags
Min: 0.0
Max: 13384586.8
Mean: 182178.41816119666
Median: 26362.82

Large Bags
Min: 0.0
Max: 5719096.61
Mean: 54332.33196537176
Median: 2647.71

XLarge Bags
Min: 0.0
Max: 551693.65
Mean: 3106.086095556407
Median: 0.0

Date unique values:
['12-27-2015' '12-20-2015' '12-13-2015' '12-6-2015' '11-29-2015'
 '11-22-2015' '11-15-2015' '11-8-2015' '11-1-2015' '10-25-2015'
 '10-18-2015' '10-11-2015' '10-4-2015' '9-27-2015' '9-20-2015' '9-13-2015'
 '9-6-2015' '8-30-2015' '8-23-2015' '8-16-2015' '8-9-2015' '8-2-2015'
 '7-26-2015' '7-19-2015' '

3.2 Inspect the earliest recorded date
- Find the earliest `Date`.
- Check if there are avocado prices recorded from the earliest date up to 2010.
- Comment: does the earliest data point look reasonable? Keep or remove?

In [13]:
# Write your code here
df['Date'] = pd.to_datetime(df['Date'])
earliest_date = df['Date'].min()
print("Earliest Date:", earliest_date)

print("Records up to 2010:")
print(df[df['Date'].dt.year <= 2010].shape)

# The earliest date 1904 is unrealistic for avocado pricing data.
# Since only 1 record exists before 2010, it is likely an error.

Earliest Date: 1904-01-21 00:00:00
Records up to 2010:
(1, 13)


3.3 Highest average price
- Find the highest value in "AveragePrice".
- Report which region it belongs to.
- Describe how you obtained the result.

In [35]:
# Write your code here
max_price = df['AveragePrice'].max()
region_max_price = df.loc[df['AveragePrice'].idxmax(), 'region']

print("Highest Average Price:", max_price)
print("Region:", region_max_price)

Highest Average Price: 3.25
Region: SanFrancisco


3.4 Highest total volume
- Find the highest total volume of avocados.
- Report which region it belongs to.
- Describe how you obtained the result.

In [34]:
# Write your code here
# Remove "TotalUS" and find highest total volume among actual regions
df_no_total = df[df['region'] != 'TotalUS']

max_volume = df_no_total['Total Volume'].max()
region_max_volume = df_no_total.loc[df_no_total['Total Volume'].idxmax(), 'region']

print("Highest Total Volume (excluding TotalUS):", max_volume)
print("Region:", region_max_volume)


Highest Total Volume (excluding TotalUS): 11274749.11
Region: West
